In [13]:
import torch
import torch.nn as nn
import math

In [ ]:
class Patchify(torch.nn.Module):
    def __init__(self, channels: int = 4, patch_size: int = 2, d_model: int = 768):
        super().__init__()
        
        # [B, C, H, W] -> [B, seq_len, d_model]
        # where seq_len = num_patch_x * num_patch_y

        self.patch_size = patch_size

        self.d_model_expander = torch.nn.Linear(channels * self.patch_size * self.patch_size, d_model)

    def forward(self, grid: torch.Tensor):
        # grid: [B, C, H, W]; post VAE 

        B, C, H, W = grid.shape
        
        grid = grid.reshape(B, C * self.patch_size * self.patch_size, H//self.patch_size, W//self.patch_size) # [B, embed_dim', num_patches_x, num_patches_y]
        grid = grid.reshape(B, C * self.patch_size * self.patch_size, -1)                                     # [B, embed_dim', num_patches_x * num_patches_y] basically [B, embed_dim', seq_len]
        grid = grid.transpose(2, 1)                                                                           # [B, seq_len, embed_dim']
        return self.d_model_expander(grid)                                                                    # [B, seq_len, embed_dim]


class Unpatchify(torch.nn.Module):
    """
    Back to pixel space from d_model dimension
    """
    def __init__(self, d_model: int = 768, channels: int = 4, patch_size: int = 2):
        super().__init__()

        self.norm   = nn.LayerNorm(d_model, elementwise_affine=False)

        self.adaLN = torch.nn.Sequential(
            torch.nn.SiLU(),
            torch.nn.Linear(d_model, 2 * d_model)
        )

        self.patch_size = patch_size
        self.d_model_shrinker = torch.nn.Linear(d_model, channels * self.patch_size * self.patch_size)       # [B, seq_len, embed_dim] -> [B, seq_len, embed_dim']


        torch.nn.init.zeros_(self.adaLN[-1].weight)
        torch.nn.init.zeros_(self.adaLN[-1].bias)
        torch.nn.init.zeros_(self.d_model_shrinker.weight)
        torch.nn.init.zeros_(self.d_model_shrinker.bias)


    def forward(self, patchified_input: torch.Tensor, context: torch.Tensor):
        shift, scale = self.adaLN(context).chunk(2, dim = -1)
        
        patchified_input = (self.norm(patchified_input) * (1 + scale.unsqueeze(1))) + shift.unsqueeze(1)
        return self.d_model_shrinker(patchified_input)


patch_class   = Patchify(channels = 4, patch_size = 2)
unpatch_class = Unpatchify(d_model = 768, channels = 4, patch_size = 2)

post_vae = torch.rand(3, 4, 32, 32)

patchified = patch_class(post_vae)
print(f"Patchified shape: {patchified.shape}")

unpatchified = unpatch_class(patchified, torch.rand(patchified.shape[0], patchified.shape[-1]))
print(f"Un-patchified shape: {unpatchified.shape}")

Patchified shape: torch.Size([3, 256, 768])
Un-patchified shape: torch.Size([3, 256, 16])


In [44]:
class Timestep_Embedder(torch.nn.Module):
    def __init__(self, timestep_freq: int, d_model: int):
        super().__init__()

        self.timestep_freq = timestep_freq

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(timestep_freq, d_model),
            torch.nn.SiLU(),
            torch.nn.Linear(d_model, d_model)
        )

    def get_timestep_embedding(self, t: torch.Tensor, dim: int):
        """
        Sinusoidal timestep embedding 
        """

        # t: [B,] -> [B, timestep_freq]

        assert dim % 2 == 0
        half = dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1)) # [half]
        args  = t.unsqueeze(-1).float() * freqs.unsqueeze(0)                                   # [B, half]
        emb   = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)                          # [B, half * 2] -> [B, timestep_freq]
        return emb
    
    def forward(self, t: torch.Tensor):
        emb = self.get_timestep_embedding(t = t, dim = self.timestep_freq)
        return self.mlp(emb)


In [46]:
timestep_embedder = Timestep_Embedder(timestep_freq = 256, d_model = 768)
t = torch.randint(0, 10, (10,))

timestep_embedder(t).shape

torch.Size([10, 768])

In [88]:
class Fourier_Embedder(torch.nn.Module):
    """
    Single number input -> embedding
    """

    def __init__(self, num_freqs = 128, d_model: int = 768):
        super().__init__()

        self.register_buffer("freqs", torch.randn(num_freqs) * 10)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(2 * num_freqs, d_model),
            torch.nn.SiLU(),
            torch.nn.Linear(d_model, d_model)
        )

    def forward(self, number):
        # number: [B, ]
        x = number.unsqueeze(1) * self.freqs.unsqueeze(0)
        x = torch.cat([torch.sin(x), torch.cos(x)], dim = -1) # [B, 2 * num_freqs]
        return self.mlp(x)                                          # [B, d_model]
    

ff_emb = Fourier_Embedder(num_freqs = 128, d_model = 768)
ff_emb(torch.randint(0, 10, (10,)) ).shape

torch.Size([10, 768])

In [83]:
torch.randint(0, 10, (10,))

tensor([6, 1, 1, 5, 5, 1, 3, 0, 5, 4])

In [18]:
dim = 768
half = dim // 2

freqs = torch.exp(-math.log(10000) * torch.arange(half) / (half - 1))
freqs.shape

torch.Size([384])

In [35]:
t = torch.randint(0, 10, (10,))
args = (t.unsqueeze(-1) * freqs.unsqueeze(0))
args.shape

torch.Size([10, 384])

In [39]:
torch.cat([torch.sin(args), torch.cos(args)], dim = -1).shape

torch.Size([10, 768])

In [52]:
class DiT_Block(torch.nn.Module):
    def __init__(self, d_model: int = 768, num_heads: int = 12):
        super().__init__()

        self.layer_norm_1 = torch.nn.LayerNorm(d_model, elementwise_affine = False) # γ, β we will predict them overselfs for different t
        self.layer_norm_2 = torch.nn.LayerNorm(d_model, elementwise_affine = False) # γ, β we will predict them overselfs for different t

        self.mha = torch.nn.MultiheadAttention(embed_dim = d_model, num_heads = num_heads, batch_first = True)

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model * 4),
            torch.nn.SiLU(),
            torch.nn.Linear(d_model * 4, d_model)
        )

        self.adaLN = torch.nn.Sequential(
            torch.nn.SiLU(),
            torch.nn.Linear(d_model, d_model * 6)
        ) 
        
        # adaLN-Zero: predict 6 modulation params from context
        # shift1, scale1  — for norm before attention
        # gate1           — gate on attention output
        # shift2, scale2  — for norm before MLP
        # gate2           — gate on MLP output

        # Zero-init the final linear so blocks start as identity
        nn.init.zeros_(self.adaLN[-1].weight)
        nn.init.zeros_(self.adaLN[-1].bias)

    
    def forward(self, patchified_inputs: torch.Tensor, context: torch.Tensor):
        # patchified_inputs: [B, seq_len, d_model]
        # context:           [B, d_model]; freq embedded + timestep embedded

        shift1, scale1, gate1, shift2, scale2, gate2 = self.adaLN(context).chunk(6, dim=-1)

        # attention block 
        patchified_inputs_norm = self.layer_norm_1(patchified_inputs)
        patchified_inputs_norm = patchified_inputs_norm * (1 + scale1.unsqueeze(1)) + shift1.unsqueeze(1)
        attn_outputs, _ = self.mha(patchified_inputs_norm, patchified_inputs_norm, patchified_inputs_norm) # self attention

        patchified_inputs +=  gate1.unsqueeze(1) * attn_outputs

        # mlp block 
        patchified_inputs_norm = self.layer_norm_1(patchified_inputs)
        patchified_inputs_norm = patchified_inputs_norm * (1 + scale2.unsqueeze(1)) + shift2.unsqueeze(1)

        patchified_inputs +=  gate2.unsqueeze(1) * self.mlp(patchified_inputs_norm)

        return patchified_inputs


In [ ]:
patch_class = Patchify(channels = 4, patch_size = 2)
post_vae = torch.rand(3, 4, 32, 32)

patchified = patch_class(post_vae)
print(f"patch shape: {patchified.shape}")

Dit_block = DiT_Block(d_model = 768, num_heads = 12)
context = torch.rand(patchified.shape[0], patchified.shape[-1])
Dit_block_out = Dit_block(patchified_inputs = patchified, context = context)

print(f"Dit Block shape: {Dit_block_out.shape}")

patch shape: torch.Size([3, 256, 768])
Dit Block shape: torch.Size([3, 256, 768])


In [56]:
print(f"{sum(p.numel() for p in Dit_block.parameters()):,}")

10,628,352
